Example notebook to explore the dataset collections provided in WP2-ICDR-O3-TC-v2

In [1]:
import xarray as xr 
import matplotlib.pyplot as plt
import os
import cartopy
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.crs as ccrs
import matplotlib as mpl
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import datetime
import cdsapi
import zipfile 
import pandas as pd
import requests

# Delivery settings

delivery_start = pd.Timestamp('01/11/2024')
delivery_end   = pd.Timestamp('30/04/2025')

collection_start = pd.Timestamp('01/01/2013')
collection_end   = pd.Timestamp('30/04/2025')

temporal_resolution = 'MS' # frequency_code = 'MS' # 'D' for daily, 'MS' for monthly, 'AS' for annual

manifest_file = 'https://wdc.dlr.de/C3S_312b_Lot2/manifest_C3S_312b_Lot2_O3_2025-10-28.txt'

parameter_list = ['total_ozone_column']

satellite  = 'METOPB' # make sure to use the correct satellite name as per the manifest file
instrument = 'GOME2' # make sure to use the correct instrument name as per the manifest file
version    = 'v0200' # make sure to use the correct version as per the manifest file



In [11]:
# Download manifest file and get list of file URLs to download corresponding to the delivery
local_fname = os.path.basename(manifest_file)
r = requests.get(manifest_file)
with open(local_fname, 'w') as file:
    file.write(r.text)

# now read the manifest file and print the lines corresponding to the satellite and instrument of interest    
with open(local_fname,'r') as file:                                                                                                                                                                                                                                             
    for line in file:
        if satellite in line and instrument in line and version in line:
            print(line)
            


http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201301-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201302-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201303-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201304-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201305-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201306-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0200.nc

http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/METOPB/GOME2/ALG/v0200/L3_MONTHLY/2013/201307-C3S-L3_OZONE-O3_PRODUCTS-GOME2-METOPB-ALG-MONTHLY-v0

In [10]:
# now a bit of manual work. copy paste an example line from above here.
# then replace the string bits corresponding to the collection (satellite, instrument, version) and the date (YYYYMM) with the appropriate variables.
# for date in dates:
#     YYYYMM = date.strftime('%Y%m')
#     fname = f'http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/{satellite}/{instrument}/ALG/{version}/L3_MONTHLY/{YYYYMM[:4]}/{YYYYMM}-C3S-L3_OZONE-O3_PRODUCTS-{instrument}-{satellite}-ALG-MONTHLY-{version}.nc'
# we will then test if all expected dates are present.

# these are the expected dates based on the collection start and end dates and the temporal resolution
dates = pd.date_range(start=delivery_start, end=delivery_end, freq=temporal_resolution) # MS: month start frequency
dates_YYYYMM = dates.strftime("%Y%m")

# read the manifest file into a dataframe and extract the date information from the file paths
df = pd.read_csv(local_fname, header=None, names=["file_path"])
# print(df.to_markdown())
df["file_date"] = df["file_path"].str.extract(r"/(\d{6})-") # the '6' assumes date is in YYYYMM format. Adjust if different.

file_dates = set(df["file_date"].dropna())
dates_set = set(dates_YYYYMM)

missing_dates = list(dates_set - file_dates)
existing_dates = list(dates_set & file_dates)

if len(missing_dates) == 0:
    print("All expected dates are present in the manifest file.")
else:
    print(f'There are {len(existing_dates)} Existing dates:', existing_dates)
    print(f'There are {len(missing_dates)} Missing dates:', missing_dates)

    

All expected dates are present in the manifest file.


In [ ]:
# Open a random file from the collection and display its contents (in this case last file in the list)
df["file_path"][len(df)-1]

'http://wdc.dlr.de/C3S_312b_Lot2/O3/TC/ASSIM/MSR/ALG/v0025/L4_MONTHLY/2024/202412-C3S-L4_OZONE-O3_PRODUCTS-MSR-ASSIM-ALG-MONTHLY-v0025.nc'